# Notebook 01 — Occurrence Ratings from Production Data

**Project:** NEXUS-FMEA (Data-Driven PFMEA + Linked Control Plan)  
**Data source:** CiP-DMD dataset via Project 1 (Sentinel-8D)  
**Purpose:** Load the per-part quality data from P1 and compute per-operation /
per-characteristic defect rates to derive AIAG-VDA Occurrence ratings.

---

**Day 1 scope:** Load the data, verify its shape, and confirm it's ready for
defect-rate computation on Day 2.

## 1 — Load Project 1 Data

In [1]:
import pandas as pd
import numpy as np

# Load raw per-part production data from Project 1 (Sentinel-8D)
df = pd.read_csv("../data/raw/parts_p1.csv")
print(f"Shape: {df.shape[0]} parts × {df.shape[1]} columns")
df.head()

Shape: 802 parts × 30 columns


,part_id_cylinder_bottom,part_id_piston_rod,assembly_rework,assembly_pressure,saw_weight,mill_surface_roughness,mill_parallelism,mill_groove_depth,mill_groove_diameter,lathe_coaxiality,...,lathe_diameter_qcpass,lathe_length_qcpass,assembly_pressure_qcpass,saw_weight_missing,lathe_coaxiality_missing,lathe_diameter_missing,lathe_length_missing,saw_anomaly_missing,mill_anomaly_missing,fail
0,103504,200102,n,9699.500,0.530674,2.364,0.0335,0.804,-0.0360,29.2,...,True,True,True,0,0,0,0,0,0,0
1,124704,200103,n,11633.167,0.552986,2.290,0.0468,0.810,-0.0410,46.2,...,True,True,True,0,0,0,0,0,0,0
2,124403,200104,n,8755.167,0.557613,2.837,0.0200,0.801,-0.0380,28.8,...,True,True,True,0,0,0,0,0,0,0
3,124301,200201,n,10119.000,0.551290,2.579,0.0446,0.795,-0.0390,14.9,...,True,True,True,0,0,0,0,0,0,0
4,124103,200202,n,12576.667,0.558249,2.683,0.0864,0.798,-0.0425,32.2,...,True,True,True,0,0,0,0,0,0,0


## 2 — Column inventory & data types

The key columns for PFMEA Occurrence are the `_qcpass` flags — one per
quality characteristic per manufacturing operation. `True` = passed QC,
`False` = defect detected.

In [2]:
# Data types overview
print("=== Data types ===")
print(df.dtypes.to_string())
print(f"\n=== Null counts ===")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "No nulls")

=== Data types ===
part_id_cylinder_bottom            int64
part_id_piston_rod                 int64
assembly_rework                      str
assembly_pressure                float64
saw_weight                       float64
mill_surface_roughness           float64
mill_parallelism                 float64
mill_groove_depth                float64
mill_groove_diameter             float64
lathe_coaxiality                 float64
lathe_diameter                   float64
lathe_length                     float64
saw_anomaly                      float64
mill_anomaly                     float64
saw_weight_qcpass                 object
mill_surface_roughness_qcpass     object
mill_parallelism_qcpass           object
mill_groove_depth_qcpass          object
mill_groove_diameter_qcpass       object
lathe_coaxiality_qcpass           object
lathe_diameter_qcpass             object
lathe_length_qcpass               object
assembly_pressure_qcpass          object
saw_weight_missing                 int

## 3 — Sanity check: per-characteristic defect counts

Quick look at how many parts failed each QC characteristic.
These counts will become Occurrence numerators on Day 2.

In [3]:
# Per-characteristic defect counts
qcpass_cols = [c for c in df.columns if c.endswith('_qcpass')]

print(f"{'Characteristic':<35} {'Total':>6} {'Pass':>6} {'Fail':>6} {'Fail %':>8}")
print("-" * 67)
for col in qcpass_cols:
    total = df[col].notna().sum()
    # Convert string 'True'/'False' to boolean if needed
    pass_count = (df[col].astype(str) == 'True').sum()
    fail_count = (df[col].astype(str) == 'False').sum()
    fail_pct = fail_count / total * 100 if total > 0 else 0
    char_name = col.replace('_qcpass', '')
    print(f"{char_name:<35} {total:>6} {pass_count:>6} {fail_count:>6} {fail_pct:>7.2f}%")

print(f"\n{'Overall part fail rate':<35} {len(df):>6} {(df['fail']==0).sum():>6} {(df['fail']==1).sum():>6} {df['fail'].mean()*100:>7.2f}%")

Characteristic                       Total   Pass   Fail   Fail %
-------------------------------------------------------------------
saw_weight                             801    801      0    0.00%
mill_surface_roughness                 801    622    179   22.35%
mill_parallelism                       801    734     67    8.36%
mill_groove_depth                      801    790     11    1.37%
mill_groove_diameter                   801    782     19    2.37%
lathe_coaxiality                       459    400     59   12.85%
lathe_diameter                         459    450      9    1.96%
lathe_length                           459    458      1    0.22%
assembly_pressure                      801    782     19    2.37%

Overall part fail rate                 802    750     52    6.48%


## 4 — Process flow: operations in the manufacturing sequence

The CiP-DMD data covers a hydraulic cylinder manufacturing process with
4 operations. This mapping will feed directly into the PFMEA process-flow
column (Day 3):

| Step | Operation | Characteristics measured |
|------|-----------|-------------------------|
| 10   | Sawing    | weight |
| 20   | Milling   | surface roughness, parallelism, groove depth, groove diameter |
| 30   | CNC Lathe | coaxiality, diameter, length |
| 40   | Assembly  | pressure |

In [4]:
# Map characteristics to manufacturing operations
# This mapping is central to the PFMEA — it ties each failure mode to its process step
OPERATION_MAP = {
    'Sawing':   ['saw_weight'],
    'Milling':  ['mill_surface_roughness', 'mill_parallelism', 'mill_groove_depth', 'mill_groove_diameter'],
    'CNC Lathe': ['lathe_coaxiality', 'lathe_diameter', 'lathe_length'],
    'Assembly': ['assembly_pressure'],
}

# Verify all qcpass columns are accounted for
mapped = [f"{c}_qcpass" for chars in OPERATION_MAP.values() for c in chars]
unmapped = set(qcpass_cols) - set(mapped)
print(f"Mapped {len(mapped)} of {len(qcpass_cols)} QC characteristics")
if unmapped:
    print(f"⚠ Unmapped: {unmapped}")
else:
    print("✅ All characteristics mapped to operations")

Mapped 9 of 9 QC characteristics
✅ All characteristics mapped to operations


## Next: Day 2

Data is loaded and verified. On Day 2 we will:
1. Compute per-operation and per-characteristic defect rates
2. Apply Wilson confidence intervals (`statsmodels`)
3. Map rates to AIAG-VDA Occurrence ratings (1–10 scale)